<a href="https://colab.research.google.com/github/Pooja-V15/AlphaL/blob/main/DAY3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from datasets import load_dataset
import pandas as pd
import numpy as np

DATASET_NAME = "DigiGreen/farmerchat-queries-v2"

try:
    # Load the dataset directly from Hugging Face
    ds = load_dataset(
        DATASET_NAME,
        "india_andhra_pradesh",
        split="train"
    )
    df = ds.to_pandas()
    print("Dataset loaded successfully!")
    print("Number of records:", len(df))
    print("Columns:", df.columns.tolist())
    display(df.head())
except Exception as e:
    print("Error loading dataset:", e)
    print("Please verify that Internet access is enabled in your notebook settings.")

README.md:   0%|          | 0.00/3.01k [00:00<?, ?B/s]

india_andhra_pradesh.parquet: reconstructing file:   0%|          |  0.00B / 4.82MB            

india_andhra_pradesh.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Dataset loaded successfully!
Number of records: 4815
Columns: ['crop', 'user_country', 'user_geo_level2', 'user_geo_level3', 'query_year_month', 'query', 'response']


,crop,user_country,user_geo_level2,user_geo_level3,query_year_month,query,response
0,generic,India,Andhra Pradesh,Visakhapatnam,2026-02,How can I control thrips in onions during the ...,Hello user! 🌱 To control thrips in onions duri...
1,generic,India,Andhra Pradesh,Sri Sathya Sai,2026-02,When is the best time to apply fertilizer for ...,Hello! [XXX]. Since your groundnuts are curren...
2,generic,India,Andhra Pradesh,Ananthapuramu,2026-02,How to properly feed a buffalo during its grow...,Feeding a growing buffalo properly is crucial ...
3,generic,India,Andhra Pradesh,Tirupati,2026-02,What are effective methods to prevent pests in...,Hello! [XXX]. Pest control is a crucial aspect...
4,generic,India,Andhra Pradesh,Ananthapuramu,2026-03,Which fertilizer is suitable for drought-resis...,Hello! [XXX]. When it comes to fertilizing dro...


In [9]:
import sys
import subprocess
import importlib.util


In [10]:
# ============================================================
# 🌿 AGRISENSE AI — AGRICULTURE CHATBOT
# Complete single-cell Kaggle Notebook implementation
# UI: Gradio | Dataset: External Hugging Face dataset
# Retrieval: TF-IDF + Cosine Similarity
# ============================================================

import sys
import subprocess
import importlib.util

# -------------------- 1. INSTALL -----------------------------

packages = {
    "gradio": "gradio",
    "datasets": "datasets",
    "sklearn": "scikit-learn"
}

for module, package in packages.items():
    if importlib.util.find_spec(module) is None:
        print(f"Installing {package}...")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package],
            capture_output=True,
            text=True,
            timeout=300
        )

        if result.returncode != 0:
            raise RuntimeError(
                f"Could not install {package}.\n"
                f"{result.stderr[-2000:]}"
            )

import gradio as gr
import re
import warnings

from datasets import load_dataset, get_dataset_config_names
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

print("Libraries ready.")

# -------------------- 2. LOAD EXTERNAL DATA ------------------

DATASET_OPTIONS = [
    "DigiGreen/farmerchat-queries",
    "DigiGreen/farmerchat-queries-v2"
]

MAX_RECORDS = 4000
records = []
dataset_name_used = None
last_error = None


def text_value(value):
    if value is None:
        return ""

    if isinstance(value, dict):
        return " ".join(text_value(v) for v in value.values())

    if isinstance(value, list):
        return " ".join(text_value(v) for v in value)

    return str(value).strip()


def locate_column(columns, options):
    normalized = {
        re.sub(r"[^a-z0-9]", "", str(c).lower()): c
        for c in columns
    }

    for option in options:
        if option in normalized:
            return normalized[option]

    for name, original in normalized.items():
        for option in options:
            if option in name:
                return original

    return None


question_options = [
    "question", "query", "userquery",
    "userquestion", "input", "prompt", "instruction"
]

answer_options = [
    "answer", "response", "output",
    "assistantresponse", "assistantanswer"
]


print("Loading agricultural data from Hugging Face...")

for ds_name in DATASET_OPTIONS:
    try:
        configs = get_dataset_config_names(ds_name)

        # Try the default configuration first, followed by others.
        configs_to_try = [None] + [
            c for c in configs if c is not None
        ]

        loaded = False

        for config in configs_to_try:
            try:
                args = {
                    "path": ds_name,
                    "split": "train",
                    "streaming": True
                }

                if config is not None:
                    args["name"] = config

                stream = load_dataset(**args)
                iterator = iter(stream)
                first = next(iterator)

                q_col = locate_column(
                    first.keys(), question_options
                )
                a_col = locate_column(
                    first.keys(), answer_options
                )

                if not q_col or not a_col:
                    continue

                temporary_records = []

                def collect(row, idx):
                    q = text_value(row.get(q_col))
                    a = text_value(row.get(a_col))

                    if len(q) >= 5 and len(a) >= 10:
                        temporary_records.append({
                            "question": q,
                            "answer": a,
                            "source": ds_name,
                            "record_id": idx
                        })

                collect(first, 0)

                for idx, row in enumerate(iterator, start=1):
                    collect(row, idx)

                    if len(temporary_records) >= MAX_RECORDS:
                        break

                    if idx >= MAX_RECORDS * 3:
                        break

                if temporary_records:
                    records = temporary_records
                    dataset_name_used = ds_name
                    loaded = True
                    break

            except Exception as e:
                last_error = e

        if loaded:
            break

    except Exception as e:
        last_error = e

if not records:
    raise RuntimeError(
        "Unable to load the external agricultural dataset.\n"
        "Check Kaggle Notebook Settings > Internet.\n"
        "If Internet is enabled, inspect the dataset repository "
        "and the final error below.\n"
        f"Last error: {last_error}"
    )

# Remove exact duplicate Q&A pairs
seen = set()
clean_records = []

for r in records:
    key = (
        r["question"].lower(),
        r["answer"].lower()
    )

    if key not in seen:
        seen.add(key)
        clean_records.append(r)

records = clean_records

print(f"Dataset: {dataset_name_used}")
print(f"Records loaded: {len(records)}")

# -------------------- 3. BUILD RETRIEVAL ---------------------

documents = [
    r["question"] + " " + r["answer"]
    for r in records
]

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=80000,
    sublinear_tf=True
)

document_matrix = vectorizer.fit_transform(documents)

print("Agricultural knowledge index is ready.")


# -------------------- 4. CHATBOT ENGINE ----------------------

def answer_question(message, history):
    message = (message or "").strip()

    if not message:
        return "Please enter an agricultural question."

    q_lower = message.lower()

    # Basic emergency and chemical safety handling
    emergency_terms = [
        "swallowed pesticide",
        "pesticide poisoning",
        "drank pesticide",
        "pesticide in eye",
        "chemical in eye",
        "inhaled pesticide"
    ]

    if any(term in q_lower for term in emergency_terms):
        return (
            "⚠️ **Safety alert**\n\n"
            "This may involve dangerous chemical exposure. "
            "Contact local emergency services or a medical "
            "professional immediately. Do not rely on this "
            "chatbot for emergency treatment instructions."
        )

    query_vector = vectorizer.transform([message])

    scores = cosine_similarity(
        query_vector, document_matrix
    ).flatten()

    top_indices = scores.argsort()[::-1][:3]
    best_idx = int(top_indices[0])
    best_score = float(scores[best_idx])

    # Avoid pretending to know answers when evidence is weak.
    if best_score < 0.12:
        return (
            "🤔 **I couldn't find sufficiently relevant evidence.**\n\n"
            "Please tell me:\n"
            "- 🌱 Which crop are you growing?\n"
            "- 📍 Which region or state are you in?\n"
            "- 🍃 What symptoms or farming problem have you noticed?\n\n"
            "I would rather ask for details than guess and give "
            "you unsuitable agricultural advice."
        )

    best_record = records[best_idx]

    response = (
        f"### 🌿 Here's what I found\n\n"
        f"{best_record['answer']}\n\n"
        f"---\n"
        f"### 📚 Evidence\n"
        f"**Dataset:** `{best_record['source']}`\n\n"
        f"**Record:** `{best_record['record_id']}`\n\n"
        f"**Text similarity:** `{best_score:.3f}`\n\n"
        f"> The similarity score measures how closely the text "
        f"matches your question. It does not establish that "
        f"the answer is correct. Verify important farming "
        f"decisions with trusted local agricultural guidance."
    )

    chemical_terms = [
        "pesticide", "insecticide", "herbicide",
        "fungicide", "chemical dosage"
    ]

    if any(term in q_lower for term in chemical_terms):
        response += (
            "\n\n⚠️ **Chemical safety:** Follow the approved "
            "product label and local regulations. Do not use "
            "an unverified recommendation as a dosage instruction."
        )

    return response


# -------------------- 5. PROFESSIONAL UI ---------------------

custom_css = """
.gradio-container {
    max-width: 1050px !important;
    margin: auto !important;
    font-family: 'Inter', sans-serif !important;
}

#hero {
    padding: 30px 24px;
    border-radius: 22px;
    background: linear-gradient(135deg, #123c2b, #24734c);
    color: white;
    text-align: center;
    margin-bottom: 18px;
}

#hero h1 {
    font-size: 38px;
    font-weight: 800;
    margin-bottom: 8px;
}

#hero p {
    font-size: 16px;
    opacity: 0.94;
}

#features {
    border: 1px solid #d8e8dc;
    border-radius: 16px;
    padding: 16px;
    background: #f4faf5;
}

#chatbox {
    border-radius: 18px;
}

button.primary {
    background: #24734c !important;
    border-color: #24734c !important;
}

footer {
    visibility: hidden;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="green",
        secondary_hue="lime",
        neutral_hue="slate"
    ),
    css=custom_css,
    title="AgriSense AI | Farming Assistant"
) as demo:

    gr.HTML("""
    <div id="hero">
        <div style="font-size:42px;">🌿</div>
        <h1>AgriSense AI</h1>
        <p>Your intelligent companion for smarter farming.</p>
        <div>
            🌱 Crop Care &nbsp; • &nbsp;
            🌾 Soil Health &nbsp; • &nbsp;
            🐛 Pest Awareness
        </div>
    </div>
    """)

    gr.HTML("""
    <div id="features">
        <b>✨ How I can help</b><br><br>
        🌱 Explore crop care &nbsp;&nbsp;
        🍃 Understand plant symptoms &nbsp;&nbsp;
        🌾 Learn about soil health<br><br>
        <small>
        Answers are retrieved from an external agricultural
        Q&A dataset. Always verify high-risk recommendations.
        </small>
    </div>
    """)

    gr.Markdown("## 💬 Chat with your farming assistant")

    gr.ChatInterface(
        fn=answer_question,
        chatbot=gr.Chatbot(
            height=420,
            layout="bubble",
            placeholder=(
                "Ask me about your crops, soil, or farming problems."
            )
        ),
        textbox=gr.Textbox(
            placeholder="e.g. Why are my tomato leaves turning yellow?",
            container=False,
            scale=7
        ),
        examples=[
            "How can I improve soil fertility?",
            "What causes yellow leaves in plants?",
            "How can farmers manage crop pests?",
            "How can I identify a disease affecting tomato plants?"
        ],
        title=None,
        description=None,
        cache_examples=False
    )

    gr.Markdown(
        """
        ---
        <center>
        🌿 <b>AgriSense AI</b> · Evidence-aware agricultural assistance
        <br>
        <small>AI-generated or retrieved information can be incorrect.
        Check local agricultural guidance before acting.</small>
        </center>
        """
    )

print("Launching AgriSense AI...")
demo.launch(share=True, debug=False)

Libraries ready.
Loading agricultural data from Hugging Face...


README.md:   0%|          | 0.00/3.01k [00:00<?, ?B/s]

Dataset: DigiGreen/farmerchat-queries
Records loaded: 4000
Agricultural knowledge index is ready.
Launching AgriSense AI...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5e0200d4387ad4c50c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
